In [ ]:
%pip install -q krauncher-jupyter

# Tutorial 10: real-time progress bar

The cell renders its own `\r`-based progress bar remotely — the relay
streams raw `\r`/`\n` and the live mirror redraws it in the cell output as
it happens. The client's own GPU metric line (`[gpu … · vram … · …s]`)
shares the same output line: metrics arrive every ~10 s and the two
overwrite each other — this notebook is the stress test for that interplay.

No flags needed: inputs/outputs are auto-detected (the weight matrix `W`
and numpy batches are non-JSON-safe and stay on the worker; the `losses`
list comes back).


In [ ]:
%load_ext krauncher_magic

In [ ]:
import os, getpass

os.environ["CAS_API_KEY"] = getpass.getpass("CAS_API_KEY: ")

In [ ]:
epochs = 5
n_batches = 40
size = 512

In [ ]:
%%krauncher
import time
import numpy as np

np.random.seed(42)
W = np.random.randn(size, size) * 0.01
bar_width = 25

losses = []
for epoch in range(1, epochs + 1):
    batch_losses = []

    for batch in range(n_batches):
        X = np.random.randn(size, size)
        y = np.random.randn(size, size)

        pred = X @ W
        loss = float(np.mean((pred - y) ** 2))
        grad = 2 * X.T @ (pred - y) / (size * size)
        W -= 0.005 * grad
        batch_losses.append(loss)

        done = batch + 1
        filled = int(bar_width * done / n_batches)
        bar = "█" * filled + "░" * (bar_width - filled)
        print(f"\rEpoch {epoch:2d}/{epochs} [{bar}] {done:2d}/{n_batches}", end="", flush=True)
        time.sleep(0.08)

    epoch_loss = sum(batch_losses) / len(batch_losses)
    losses.append(epoch_loss)
    bar = "█" * bar_width
    print(f"\rEpoch {epoch:2d}/{epochs} [{bar}] {n_batches}/{n_batches}  loss={epoch_loss:.6f}")


In [ ]:
print(f"Initial loss: {losses[0]:.6f}")
print(f"Final loss:   {losses[-1]:.6f}")
print(f"Improvement:  {(losses[0] - losses[-1]) / losses[0] * 100:.2f}%")
